# 02 - Splits, leakage and the two sensitivity sweeps

Lock the evaluation protocol before anything is trained, and price the two
design choices currently being assumed.

**Protocol, borrowed from LineageFlow:** components as the splitting unit,
threshold sensitivity, censoring discipline, and every score reported as a
mean +/- sd over repeats with a permutation null.

Only rung 1 (five state fractions) is scored here - it is the one predictor
that needs no expression matrix. Rungs for pseudobulk mean and the kernel mean
embedding slot into the same harness once the matrix is converted, by swapping
`X` and leaving everything else alone.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent / "src"))
import etracer as et

P = et.Paths(Path.cwd().parent)
manifest = pd.read_csv(P.derived / "clone_manifest.csv")
Xcols = [f"state_{s}" for s in et.STATES]

In [ ]:
# --- the split, and proof that it holds -----------------------------------
# Grouped by clone so no clone appears in train and test. Notebook 00 proved
# components == clones for this deposit, so the clone id is a valid group key.
def build(extinct="pseudo", min_bl_cells=20, stage_dst="EE"):
    g = et.growth_table(manifest, "BL", stage_dst, extinct=extinct)
    g = g[g.n_cells_src >= min_bl_cells]
    feat = et.state_features(manifest, "BL")
    d = (g.groupby(["clone", "family"])["log2_growth"].mean().rename("y").reset_index()
           .merge(feat, on=["clone", "family"], how="inner").dropna(subset=["y"]))
    return d[Xcols].to_numpy(), d["y"].to_numpy(), d["clone"].to_numpy(), d

X, y, grp, d = build()
print(f"design {X.shape}, {len(np.unique(grp))} clones, {len(d)} clone-family units")

rng = np.random.default_rng(0)
for k, (tr, te) in enumerate(et.grouped_folds(grp, 5, rng)):
    overlap = set(grp[tr]) & set(grp[te])
    print(f"  fold {k}: train {tr.sum():4d}  test {te.sum():4d}  shared clones {len(overlap)}")
    assert not overlap, "clone leaked across the split"
print("no clone appears on both sides of any fold")

In [ ]:
# --- headline number, with its null --------------------------------------
scores = et.grouped_cv_r(X, y, grp, n_repeats=20, seed=0)
null = et.permutation_null_r(X, y, grp, n_perm=20)
base = et.report("rung 1: 5 state fractions", scores, null)
print(pd.Series(base).to_string())
print(f"\nper-repeat: {np.nanmin(scores):.3f} .. {np.nanmax(scores):.3f}")

In [ ]:
# --- SWEEP A: minimum BL cells per clone ---------------------------------
# Doubles as the Bonham-Carter mitigation: the proliferation bias in multi-time
# clones gets worse as the capture x barcode rate falls, so a score that only
# holds at one threshold is not a result.
rows = []
for t in [5, 10, 20, 50]:
    Xi, yi, gi, di = build(min_bl_cells=t)
    if len(np.unique(gi)) < 10:
        print(f"  threshold {t}: too few clones, skipped"); continue
    s = et.grouped_cv_r(Xi, yi, gi, n_repeats=20, seed=0)
    rows.append(dict(sweep="min_bl_cells", value=t, n_units=len(yi),
                     n_clones=len(np.unique(gi)),
                     r_mean=np.nanmean(s), r_sd=np.nanstd(s)))
    print(f"  >= {t:3d} BL cells: {len(yi):4d} units, {len(np.unique(gi)):3d} clones, "
          f"r = {np.nanmean(s):.3f} +/- {np.nanstd(s):.3f}")
sweep_a = pd.DataFrame(rows)

In [ ]:
# --- SWEEP B: extinction as observed zero vs censored --------------------
# These two positions cannot both be right. Flooring extinction at a
# pseudo-count is where most of the log2 dynamic range comes from; censoring
# refuses to read death from absence. Report both, always.
rows = []
for mode in ["pseudo", "censor"]:
    Xi, yi, gi, di = build(extinct=mode)
    s = et.grouped_cv_r(Xi, yi, gi, n_repeats=20, seed=0)
    rows.append(dict(sweep="extinct", value=mode, n_units=len(yi),
                     n_clones=len(np.unique(gi)),
                     y_range=float(np.nanmax(yi) - np.nanmin(yi)),
                     r_mean=np.nanmean(s), r_sd=np.nanstd(s)))
    print(f"  extinct={mode:7s}: {len(yi):4d} units, y range {rows[-1]['y_range']:5.1f}, "
          f"r = {np.nanmean(s):.3f} +/- {np.nanstd(s):.3f}")
sweep_b = pd.DataFrame(rows)

# How much of the signal is carried by the extinct clones alone?
_, _, _, dp = build(extinct="pseudo")
gp = et.growth_table(manifest, "BL", "EE", extinct="pseudo")
gp = gp[gp.n_cells_src >= 20]
print(f"\nEE-undetected rows: {(~gp.detected_dst).sum()} / {len(gp)} "
      f"({(~gp.detected_dst).mean():.1%})")
print("If r collapses under censoring, the primary target definition changes.")

In [ ]:
# --- SWEEP C: family hold-out, with the rung-0 lookup alongside ----------
# On a family hold-out the clone-identity lookup is available, so it must be
# reported next to every expression arm. n = 3 folds: directional only.
gm = (et.growth_table(manifest, "BL", "EE", extinct="pseudo")
        .query("n_cells_src >= 20")
        .groupby(["clone", "family"])["log2_growth"].mean().rename("y").reset_index())
dd = gm.merge(et.state_features(manifest, "BL"), on=["clone", "family"], how="inner")

rows = []
for held in sorted(dd.family.unique()):
    tr, te = dd[dd.family != held], dd[dd.family == held]
    # rung 1: ridge on state fractions
    p1 = et.ridge_predict(tr[Xcols].to_numpy(), tr["y"].to_numpy(),
                          te[Xcols].to_numpy(), alpha=10.0)
    # rung 0: this clone's mean growth in the OTHER families
    lookup = tr.groupby("clone")["y"].mean()
    p0 = te["clone"].map(lookup).to_numpy()
    cov = np.isfinite(p0).mean()
    rows.append(dict(held_out=held, n_test=len(te),
                     rung0_lookup_r=et.pearson(p0, te["y"].to_numpy()),
                     rung0_coverage=cov,
                     rung1_state_r=et.pearson(p1, te["y"].to_numpy())))
    print(f"  hold out {held}: n={len(te):3d}  rung0 r={rows[-1]['rung0_lookup_r']:.3f} "
          f"(covers {cov:.0%})  rung1 r={rows[-1]['rung1_state_r']:.3f}")
sweep_c = pd.DataFrame(rows)
print(f"\nmean rung0 {sweep_c.rung0_lookup_r.mean():.3f} | "
      f"mean rung1 {sweep_c.rung1_state_r.mean():.3f}")
print("If rung 0 wins here, identity transfers across families and the "
      "held-out-CLONE split is the only one that tests the expression claim.")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13.5, 4))

ax[0].errorbar(sweep_a.value, sweep_a.r_mean, yerr=sweep_a.r_sd, marker="o", capsize=4)
ax[0].axhline(np.nanmean(null), color="grey", ls=":", label="permutation null")
ax[0].set_xlabel("minimum BL cells per clone"); ax[0].set_ylabel("out-of-fold r")
ax[0].set_title("A. threshold sensitivity"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.25)

ax[1].bar(sweep_b.value, sweep_b.r_mean, yerr=sweep_b.r_sd, capsize=5, alpha=0.85)
ax[1].axhline(np.nanmean(null), color="grey", ls=":")
ax[1].set_ylabel("out-of-fold r"); ax[1].set_title("B. extinction: zero vs censored")
ax[1].grid(alpha=0.25, axis="y")

w = 0.35; xs = np.arange(len(sweep_c))
ax[2].bar(xs - w/2, sweep_c.rung0_lookup_r, w, label="rung 0 lookup", alpha=0.85)
ax[2].bar(xs + w/2, sweep_c.rung1_state_r, w, label="rung 1 states", alpha=0.85)
ax[2].set_xticks(xs); ax[2].set_xticklabels(sweep_c.held_out)
ax[2].set_ylabel("r on held-out family"); ax[2].set_title("C. family hold-out (n=3)")
ax[2].legend(fontsize=8); ax[2].grid(alpha=0.25, axis="y")

plt.tight_layout(); plt.show()

In [ ]:
# --- save ----------------------------------------------------------------
allsweeps = pd.concat([sweep_a, sweep_b], ignore_index=True)
allsweeps.to_csv(P.derived / "sweeps.csv", index=False)
sweep_c.to_csv(P.derived / "family_holdout.csv", index=False)
pd.DataFrame([base]).to_csv(P.derived / "rung1_headline.csv", index=False)
print(allsweeps.to_string(index=False))

## Decision gates

1. **Threshold.** If r moves materially across 5 -> 50 BL cells, the headline
   number needs a threshold stated beside it, and the writeup reports the range
   rather than one value.
2. **Censoring.** If the ranking flips between `pseudo` and `censor`, the
   primary target definition has to be settled before notebook 06.
3. **Family hold-out.** If rung 0 beats rung 1 here, say so plainly: identity
   transfers, and only the held-out-clone split tests whether expression adds
   anything.

## Adding the expression rungs later

Once `derived/` has an expression matrix, the two remaining rungs reuse this
harness unchanged - build `X` and call the same `grouped_cv_r`:

```python
# rung 1b: pseudobulk mean over a fate-blind gene panel
X = np.vstack([expr[cells_of(clone, family)].mean(0) for ...])

# rung 2: kernel mean embedding of the clone's cell distribution
#   mu_k = (1/n_k) sum_i phi(x_i)   with phi = random Fourier features
X = np.vstack([rff(expr[cells_of(clone, family)]).mean(0) for ...])
```

Keep the gene panel fate-blind and select it **inside** each training fold.